In [14]:
from collections import Counter
import re
from pathlib import Path

# Folder containing your .md files and subfolders
folder_path = Path("/Users/elmarmammadov/Documents/NLP/CleanTextOCR")

text = ""

# Recursively loop over all .md files in the folder and subfolders
for file_path in folder_path.rglob("*.md"):
    with open(file_path, "r", encoding="utf-8") as f:
        text += f.read()
        text += " "  # separate files by space

print(f"Total length of text: {len(text)} characters")
dot_patterns = re.findall(r'\S*\.\S*', text)
Counter(dot_patterns).most_common(20)

Total length of text: 57231566 characters


[('idi.', 10660),
 ('var.', 5888),
 ('.', 4314),
 ('yoxdur.', 4053),
 ('oldu.', 3966),
 ('etdi.', 3241),
 ('edir.', 3110),
 ('dedi.', 3076),
 ('olur.', 3030),
 ('gəlir.', 2536),
 ('bəy.', 2335),
 ('deyil.', 2308),
 ('olsun.', 2068),
 ('başladı.', 1951),
 ('gəldi.', 1919),
 ('olar.', 1906),
 ('olmaz.', 1887),
 ('mən.', 1883),
 ('getdi.', 1864),
 ('edirdi.', 1772)]

In [15]:
import nltk
from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktTrainer

trainer = PunktTrainer()
trainer.train(open("/Users/elmarmammadov/Documents/NLP/CleanTextOCR/Cafa Cabbarli/Eserleri_I_cild.md").read())
tokenizer = PunktSentenceTokenizer(trainer.get_params())

sentences = tokenizer.tokenize(text)

In [16]:
gold_boundaries = set()

current_pos = 0
for sent in sentences:
    end_pos = text.find(sent, current_pos) + len(sent)
    last_dot = text.rfind(".", current_pos, end_pos)
    
    if last_dot != -1:
        gold_boundaries.add(last_dot)
    
    current_pos = end_pos

In [17]:
import string

tokens = re.findall(r'\w+|[^\w\s]', text)
dot_positions = [m.start() for m in re.finditer(r'\.', text)]
data = []
dot_iter = iter(dot_positions)
window = 1
current_dot_pos = next(dot_iter, None)
letters_upper = [
    "A", "B", "C", "Ç", "D", "E", "Ə", "F", "G", "Ğ",
    "H", "X", "I", "İ", "J", "K", "Q", "L", "M", "N",
    "O", "Ö", "P", "R", "S", "Ş", "T", "U", "Ü", "V",
    "Y", "Z"
]

char_pointer = 0

for i, token in enumerate(tokens):
    if token == ".":
        dot_pos = current_dot_pos
        
        left_context = tokens[max(0, i-window):i]
        right_context = tokens[i+1:i+1+window]
        
        label = 1 if dot_pos in gold_boundaries else 0
        if left_context and left_context[-1] == ".":
            stripped = right_context[0].lstrip()
            if stripped and stripped[0] in letters_upper:
                label = 1
                
        
        last_token = left_context[-1].rstrip(string.punctuation)

        if last_token and last_token[-1].isdigit() and right_context:
            stripped = right_context[0].lstrip()
            if stripped and stripped[0] in string.punctuation:
                label = 0
                print(left_context, right_context)
        data.append({
            "left": left_context,
            "right": right_context,
            "label": label
        })
        
        current_dot_pos = next(dot_iter, None)

['0'] ['.']
['8'] ['.']
['8'] ['.']
['8'] ['.']
['71'] ['.']
['7055'] [')']
['1'] ['-']
['80'] ['.']
['5'] [')']
['ӘХ0'] ['.']
['5'] [')']
['5'] [')']
['5'] [')']
['5'] [')']
['22'] ['.']
['2000'] ['-']
['8'] ['.']
['3'] ['.']
['8'] ['.']
['3'] ['.']
['3'] ['.']
['9'] ['.']
['17'] ['=']
['5121'] ['.']
['1961'] ['.']
['4ед1К1әг1'] ['.']
['9'] ['.']
['1'] ['.']
['рӧгтиӧгздп622'] ['.']
['20'] ['.']
['5'] [')']
['4'] [')']
['1940'] ['.']
['0000000'] ['.']
['даудајаг1'] ['.']
['2'] ['.']
['аИәз1'] ['.']
['2'] ['.']
['0090000000'] ['.']
['3'] ['.']
['2'] ['/']
['0'] ['.']
['3'] ['.']
['3'] ['.']
['9999000'] ['.']
['3'] ['.']
['9'] ['.']
['0'] ['.']
['9'] ['.']
['0000000'] ['.']
['99'] ['.']
['3'] ['.']
['9'] ['.']
['3'] ['.']
['3'] ['.']
['3'] ['.']
['9090000800'] ['.']
['9'] ['.']
['3'] ['.']
['аӱ1'] ['.']
['22'] ['.']
['57'] ['.']
['hesabsızdırlar1'] ['.']
['Кӧјәә51'] ['.']
['а1ә51'] ['.']
['551211'] ['.']
['абасј1аг1'] ['.']
['ва11'] ['.']
['76'] ['.']
['10'] ['.']
['5122'] ['.']
['ә1'] [

In [ ]:
import pandas as pd

rows = []

for row in data:
    rows.append({
        "left_context": " ".join(row["left"]),
        "right_context": " ".join(row["right"]),
        "label": row["label"]
    })

df = pd.DataFrame(rows)
df.to_csv("sentence_boundary_data.csv", index=False)